# A9 · Opening up the model, part 1: why does it say what it says?

**Where we are**

```
A6–A7  trained the model (LightGBM)   →   A8  tested it once on Jul–Aug: 86% on time at P75
A9     HERE, part 1: why does it say what it says?   (part 2: is it fair to every group?  part 3: tuning + model card)
```

**The one idea:** every prediction is **a starting point plus one push per column**.

- **Starting point:** the model's average prediction, the same number for every visit.
- **Push:** how many minutes one column moves *this* visit's prediction, up (+) or down (−).
- The pushes **add up exactly** to the prediction. These pushes are called **SHAP values**; LightGBM computes them itself with `predict(X, pred_contrib=True)`.

**Why we can do something real projects can't:** we wrote the simulator (A2), so we know the true rule. We can check whether a push the model found matches the push the rule says it should be.

We explain the **final P80 model** from A8 (trained on January–June) on the **July–August test visits**. Explaining doesn't choose anything, so looking at the test months here is fine.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from slotpred import explain as ex
from slotpred.config import ROOT

booster = ex.final_model(0.8)          # the final P80 model from A8
X, visits = ex.test_table()            # Jul-Aug: the model's columns, and the visits with the simulator's answer key
print(X.shape, "columns:", list(X.columns))

## 1 · Split every prediction into pushes

**TODO(you):** write `pushes(booster, X)` in [slotpred/explain.py](../slotpred/explain.py). It's about three lines: call `booster.predict(X, pred_contrib=True)` and put the result in a DataFrame with the right column names and index. The docstring says exactly what to return. Then run the tests below.

In [ ]:
!"{sys.executable}" -m pytest "{ROOT / 'tests' / 'test_explain.py'}" -q -k pushes

In [ ]:
P = ex.pushes(booster, X)
print("starting point:", round(P["start"].iloc[0], 2), "min")
print("largest gap between (sum of pushes) and the prediction:", (P.sum(axis=1) - booster.predict(X)).abs().max())

## 2 · One visit, column by column

Dr Deepak Rao, **follow-up by video**, HSR, age 46, no prior reports, Tuesday 15:45.

In [ ]:
i = visits.index[(visits.doctor == "Deepak Rao") & (visits.visit_type == "follow-up") & (visits.teleconsult == 1)
                 & (visits.clinic == "HSR") & (visits.age == 46)][0]
row = P.loc[i].drop("start").sort_values(key=abs, ascending=False)
print(f"start {P.loc[i, 'start']:.1f}")
for col, push in row.items():
    print(f"  {col:14s} {push:+.2f}")
print(f"= P80 {booster.predict(X.loc[[i]])[0]:.2f} min   (actual: {visits.loc[i, 'minutes']:.1f} min)")

## 3 · Which columns matter, on average?

Average the *size* of each column's push over all test visits (ignoring up or down).

In [ ]:
avg = ex.average_push(P)
print(avg.round(2))
avg[::-1].plot.barh(figsize=(6, 3.5), title="average size of each column's push (minutes), P80")
plt.xlabel("minutes"); plt.tight_layout(); plt.show()

## 4 · Did the model find the video rule?

The hidden rule (A2): video multiplies the visit's minutes by **0.80**. So for a visit that takes `m` minutes in person, video's true push on the P80 is

```
m × 0.80 × 1.217  −  m × 1.217   =   m × (0.80 − 1) × 1.217
```

where 1.217 = e^(−σ²/2 + 0.8416 σ) is A2's P80 factor (σ = 0.28).

**TODO(you):** write `video_push_by_rule(in_person_minutes, q)` in [slotpred/explain.py](../slotpred/explain.py), for any q (use `norm.ppf(q)` for the z-score). Then run the tests.

In [ ]:
!"{sys.executable}" -m pytest "{ROOT / 'tests' / 'test_explain.py'}" -q -k video

In [ ]:
video = visits["teleconsult"].astype(bool).to_numpy()
model_push = P["teleconsult"].to_numpy()[video]
rule_push = ex.video_push_by_rule(visits["in_person_minutes"].to_numpy()[video], 0.8)
print(f"{video.sum()} video visits in Jul-Aug ({video.mean():.1%} of all visits)")
print(f"average push   model {model_push.mean():.2f} min   rule {rule_push.mean():.2f} min")
print(f"the example visit: model {P.loc[i, 'teleconsult']:.2f}, rule {ex.video_push_by_rule(visits.loc[i, 'in_person_minutes'], 0.8):.2f}")

plt.figure(figsize=(5, 5))
plt.scatter(rule_push, model_push, s=8, alpha=0.4)
lim = [min(rule_push.min(), model_push.min()) - 0.2, 0]
plt.plot(lim, lim, "k--", lw=1, label="model = rule")
plt.xlabel("push from the hidden rule (min)"); plt.ylabel("push the model found (min)")
plt.title("Video consults, P80"); plt.legend(); plt.tight_layout(); plt.show()

**Questions** (short answers, your own words)
1. General Medicine is one of the quickest departments, yet in section 2 `department` got only about **−0.06** while `doctor` got **−2.8**. Where did the department's credit go?
2. In section 3, `teleconsult` is a tiny push on average, yet for the visit in section 2 it was about **−1.5**. How can both be true?
3. The visit in section 2 actually took **4.2 minutes**, against a P80 of 6.6. Was the model wrong for this visit?
4. In the scatter, short video visits sit on the dashed line, but for long ones the model's push is smaller than the rule's. Look at who gets video consults in `simulate.py` (which visit types?). Why might the model have learned video as "about 1.5–2 minutes off" instead of "20% off"?

*Answers:*